# 🗺️ Python 2D Dynamic Programming — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> 2D DP is like filling in a crossword grid from top-left to bottom-right.
> Each cell's answer depends on the cells above it, to the left of it, or diagonally.
> You never recompute — just look up what you already filled in earlier.
> The rows represent one dimension (often one string or row), the columns represent another.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is 2D DP? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Patterns](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Unique Paths (LC 62)](#5) |
| 6 | [Pattern 2: Unique Paths with Obstacles (LC 63)](#6) |
| 7 | [Pattern 3: Edit Distance (LC 72)](#7) |
| 8 | [Pattern 4: Longest Common Subsequence (LC 1143)](#8) |
| 9 | [The 2D DP Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is 2D DP? The Visual Model

```
               2D DP — THE CROSSWORD GRID BUILDER

  Example: Unique Paths on a 3×3 grid
  dp[i][j] = number of ways to reach cell (i,j) from (0,0)

       j=0   j=1   j=2
  i=0 [ 1 ] [ 1 ] [ 1 ]   ← top row: only 1 way (go right)
  i=1 [ 1 ] [ 2 ] [ 3 ]   ← dp[1][1] = dp[0][1] + dp[1][0] = 1+1 = 2
  i=2 [ 1 ] [ 3 ] [ 6 ]   ← dp[2][2] = dp[1][2] + dp[2][1] = 3+3 = 6
                                          ↑             ↑
                                       from above   from left

  RECURRENCE: dp[i][j] = dp[i-1][j] + dp[i][j-1]
  BASE CASE:  dp[0][j] = 1 for all j  (top row)
              dp[i][0] = 1 for all i  (left column)

  TWO-STRING PATTERN (LCS, Edit Distance):
  One string on rows, other string on columns.
  dp[i][j] = best answer considering first i chars of s1 and first j chars of s2.

       ''  'a' 'b' 'c'
  ''  [ 0]  [1]  [2]  [3]
  'a' [ 1]  [0]  [1]  [2]
  'b' [ 2]  [1]  [0]  [1]
  'c' [ 3]  [2]  [1]  [0]
  (Edit distance table for 'abc' vs 'abc')
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# BASIC 2D DP GRID SETUP

# Option 1: full 2D dp array
rows, cols = 3, 4
dp = [[0] * cols for _ in range(rows)]  # rows x cols grid of zeros
print("empty dp grid:")
for row in dp:
    print(row)

# Option 2: dp with padding (extra row/col of 0 simplifies base cases)
# used for string problems where i=0 or j=0 represents empty string
m, n = 3, 4   # string lengths
dp_padded = [[0] * (n + 1) for _ in range(m + 1)]  # (m+1) x (n+1)
print(f"\npadded dp grid (m+1={m+1}, n+1={n+1}):")
for row in dp_padded:
    print(row)

# Option 3: 1D rolling array space optimization (save O(m*n) → O(n))
# dp[j] represents the current row; prev[j] holds the previous row
dp_1d = [0] * (n + 1)  # only keep one row at a time
print("\n1D rolling array:", dp_1d)
print("2D DP setups demonstrated.")

<a id='3'></a>
## 3. The Core API — All Patterns

```
2D DP PATTERN          RECURRENCE                      TYPICAL PROBLEMS
────────────────────────────────────────────────────────────────────────
Grid paths             dp[i][j] = dp[i-1][j]+dp[i][j-1]   LC 62, 63
Two strings (LCS)      match: dp[i-1][j-1]+1               LC 1143
                       no match: max(dp[i-1][j], dp[i][j-1])
Edit distance          match: dp[i-1][j-1]                 LC 72
                       no match: 1+min(del,ins,replace)
Palindrome DP          dp[i][j] = dp[i+1][j-1]+2 if match  LC 5, 516
                       else max(dp[i+1][j], dp[i][j-1])

FILL ORDER:
  Grid paths:     top-left → bottom-right (row by row)
  Two strings:    i=1..m, j=1..n (outer=rows, inner=cols)
  Palindrome:     diagonal — length 1, then 2, then 3, ...

THINGS YOU DO NOT DO:
❌  Define dp[i][j] ambiguously — write it out in English first
❌  Forget to initialize the first row and column as base cases
❌  Use dp[i][j] when you meant dp[i-1][j] — off-by-one in padded arrays
❌  Confuse 'delete from s1' vs 'insert into s1' — they are symmetric
```

In [ ]:
# Demo: print a filled 2D DP table for Unique Paths (3x4)
def build_unique_paths_table(rows, cols):
    dp = [[1] * cols for _ in range(rows)]  # base: top row and left col = 1
    for i in range(1, rows):
        for j in range(1, cols):
            dp[i][j] = dp[i-1][j] + dp[i][j-1]  # from above + from left
    return dp

table = build_unique_paths_table(3, 4)
print("Unique Paths DP table (3x4):")
for row in table:
    print(row)
print("answer (bottom-right):", table[2][3])  # 10
print("2D DP table demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"unique paths in grid"                  dp[i][j] = dp[i-1][j] + dp[i][j-1]
"grid with obstacles"                   same + if obstacle: dp[i][j]=0
"minimum edit distance"                 dp[i][j] = min(del, ins, replace)
"longest common subsequence"            match → dp[i-1][j-1]+1, else max
"longest palindromic subsequence"       LCS of s and reversed s
"longest palindromic substring"         expand from center (not DP usually)
"two strings, align/transform"          edit distance shape
"minimum path sum in grid"              dp[i][j] = min(above, left) + grid[i][j]
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Unique Paths — LC 62

---

```
PROBLEM:
  A robot is at top-left of an m×n grid. It can only move right or down.
  Count the number of unique paths to reach the bottom-right corner.

TRICK:
  dp[i][j] = number of ways to reach cell (i,j).
  You can only arrive from above (i-1,j) or from the left (i,j-1).
  Base: entire top row and left column = 1 (only one way to reach them).
  Fill row by row from top-left to bottom-right.

SLOW MOTION TRACE on 3×3:

       j=0  j=1  j=2
  i=0 [ 1 ] [ 1 ] [ 1 ]   base row
  i=1 [ 1 ] [ 2 ] [ 3 ]   dp[1][1]=dp[0][1]+dp[1][0]=1+1=2
  i=2 [ 1 ] [ 3 ] [ 6 ]   dp[2][2]=dp[1][2]+dp[2][1]=3+3=6

  answer = dp[2][2] = 6

KEY INSIGHT:
  You can also solve this with combinatorics: C(m+n-2, m-1).
  But DP generalizes to obstacles and weighted grids — prefer DP.

TIME:  O(m*n)
SPACE: O(n) with 1D rolling array optimization
```

In [ ]:
def unique_paths(m, n):
    """
    LC 62 — Unique Paths
    Approach: dp[i][j] = paths from above + from left; 1D rolling array for O(n) space.
    Args:
        m (int): number of rows.
        n (int): number of columns.
    Returns:
        int: number of unique paths from top-left to bottom-right.
    Time:  O(m*n) — fill every cell once
    Space: O(n)   — rolling 1D array (current row only)
    """
    dp = [1] * n          # base: first row is all 1s (can only go right)
    for _ in range(1, m): # for each subsequent row
        for j in range(1, n):
            dp[j] += dp[j-1]  # dp[j] (from above) += dp[j-1] (from left)
            # dp[j] still holds the value from the previous row (= from above)
            # dp[j-1] was just updated this row (= from left)
    return dp[n-1]

# Slow motion on m=3, n=3:
# init dp=[1,1,1]  (first row)
# row i=1:
#   j=1: dp[1]+=dp[0] → dp[1]=1+1=2  → dp=[1,2,1]
#   j=2: dp[2]+=dp[1] → dp[2]=1+2=3  → dp=[1,2,3]
# row i=2:
#   j=1: dp[1]+=dp[0] → dp[1]=2+1=3  → dp=[1,3,3]
#   j=2: dp[2]+=dp[1] → dp[2]=3+3=6  → dp=[1,3,6]
# return dp[2]=6

def test_harness(fn):
    tests = [
        (3, 7, 28),
        (3, 3, 6),
        (1, 1, 1),
        (2, 2, 2),
        (7, 3, 28),
        (3, 2, 3),
    ]
    passed = 0
    for *inputs, expected in tests:
        m, n = inputs
        got = fn(m, n)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | m={m} n={n} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(unique_paths)
print("unique_paths defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Unique Paths with Obstacles — LC 63

---

```
PROBLEM:
  Same as LC 62, but some cells are obstacles (1 = blocked, 0 = open).
  Blocked cells are unreachable — dp[i][j] = 0 there.

TRICK:
  Same DP as LC 62, but:
  - Before filling dp[i][j], check if obstacle_grid[i][j] == 1 → set dp[i][j] = 0.
  - Top-left = 1 only if obstacle_grid[0][0] == 0.
  - In the base row/column, stop at first obstacle (cells beyond it are 0).

SLOW MOTION TRACE on:
  [[0,0,0],
   [0,1,0],
   [0,0,0]]

       j=0  j=1  j=2
  i=0 [ 1 ] [ 1 ] [ 1 ]   base row (no obstacles)
  i=1 [ 1 ] [ 0 ] [ 1 ]   obstacle at (1,1) → 0; dp[1][2]=dp[0][2]+0=1
  i=2 [ 1 ] [ 1 ] [ 2 ]   dp[2][2]=dp[1][2]+dp[2][1]=1+1=2

  answer = 2

KEY INSIGHT:
  An obstacle zeroes out that cell AND propagates zero into cells that
  only have paths passing through the obstacle.

TIME:  O(m*n)
SPACE: O(n)
```

In [ ]:
def unique_paths_with_obstacles(obstacle_grid):
    """
    LC 63 — Unique Paths II
    Approach: Same as LC 62 but set dp[i][j]=0 wherever obstacle_grid[i][j]==1.
    Args:
        obstacle_grid (List[List[int]]): 0=open, 1=obstacle.
    Returns:
        int: number of unique paths avoiding obstacles.
    Time:  O(m*n) — one pass through the grid
    Space: O(n)   — 1D rolling array
    """
    m, n = len(obstacle_grid), len(obstacle_grid[0])

    if obstacle_grid[0][0] == 1 or obstacle_grid[m-1][n-1] == 1:
        return 0   # start or end is blocked — no paths possible

    dp = [0] * n
    dp[0] = 1      # start cell = 1 way (if not blocked, already checked above)

    # fill first row base case — stop propagating at first obstacle
    for j in range(1, n):
        dp[j] = 0 if obstacle_grid[0][j] == 1 else dp[j-1]

    for i in range(1, m):
        # update left column first — it can only be reached from above
        dp[0] = 0 if obstacle_grid[i][0] == 1 else dp[0]
        for j in range(1, n):
            if obstacle_grid[i][j] == 1:
                dp[j] = 0           # obstacle — no paths through here
            else:
                dp[j] += dp[j-1]    # from above (dp[j] unchanged) + from left (dp[j-1])

    return dp[n-1]

# Slow motion on [[0,0,0],[0,1,0],[0,0,0]]:
# dp init=[1,0,0] → first row: dp=[1,1,1]
# i=1: dp[0]=1; j=1: obstacle → dp[1]=0; j=2: dp[2]+=dp[1]=1+0=1 → dp=[1,0,1]
# i=2: dp[0]=1; j=1: dp[1]+=dp[0]=0+1=1; j=2: dp[2]+=dp[1]=1+1=2 → dp=[1,1,2]
# return dp[2]=2

def test_harness(fn):
    tests = [
        ([[0,0,0],[0,1,0],[0,0,0]], 2),
        ([[0,1],[0,0]], 1),
        ([[1,0]], 0),       # start blocked
        ([[0,0],[1,0]], 1), # one path via top
        ([[0,0,0],[0,0,0],[0,0,0]], 6),  # no obstacles
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(unique_paths_with_obstacles)
print("unique_paths_with_obstacles defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Edit Distance — LC 72

---

```
PROBLEM:
  Given two strings word1 and word2, return the minimum number of operations
  (insert, delete, replace) to convert word1 into word2.

TRICK:
  dp[i][j] = min edits to convert word1[0..i-1] to word2[0..j-1].
  If characters match: dp[i][j] = dp[i-1][j-1]  (no edit needed)
  If they differ:      dp[i][j] = 1 + min(
                         dp[i-1][j],    ← delete from word1
                         dp[i][j-1],    ← insert into word1 (delete from word2)
                         dp[i-1][j-1]   ← replace
                       )
  Base: dp[i][0]=i (delete i chars), dp[0][j]=j (insert j chars)

SLOW MOTION TRACE on word1="horse", word2="ros":

       ''   r    o    s
  ''  [ 0]  [1]  [2]  [3]
  h   [ 1]  [1]  [2]  [3]  h≠r: 1+min(0,1,1)=1
  o   [ 2]  [2]  [1]  [2]  o==o: dp[1][1]=1; o≠r: 1+min(1,2,1)=2
  r   [ 3]  [2]  [2]  [2]  r==r: dp[1][0]+? ; r≠o: ...
  s   [ 4]  [3]  [3]  [2]  s==s: dp[3][2]=2
  e   [ 5]  [4]  [4]  [3]

  answer = dp[5][3] = 3 (delete 'h', replace 'r' with nothing, delete 'e')

KEY INSIGHT:
  The three options (del, ins, replace) correspond to moving from
  dp[i-1][j], dp[i][j-1], dp[i-1][j-1] respectively in the table.

TIME:  O(m*n)
SPACE: O(n) with 1D rolling optimization
```

In [ ]:
def min_distance(word1, word2):
    """
    LC 72 — Edit Distance
    Approach: 2D DP — dp[i][j] = min edits to convert word1[:i] to word2[:j].
    Args:
        word1 (str): source string.
        word2 (str): target string.
    Returns:
        int: minimum number of insert/delete/replace operations.
    Time:  O(m*n) — m=len(word1), n=len(word2)
    Space: O(n)   — 1D rolling array
    """
    m, n = len(word1), len(word2)

    # dp[j] = edits to convert word1[:i] to word2[:j]
    dp = list(range(n + 1))   # base: dp[0][j] = j (need j insertions)

    for i in range(1, m + 1):
        prev = dp[0]           # save dp[i-1][j-1] before overwriting
        dp[0] = i              # base: dp[i][0] = i (need i deletions)
        for j in range(1, n + 1):
            temp = dp[j]       # save dp[i-1][j] before we overwrite it
            if word1[i-1] == word2[j-1]:
                dp[j] = prev   # characters match — no edit needed (take diagonal)
            else:
                dp[j] = 1 + min(
                    temp,      # delete from word1   (from above)
                    dp[j-1],   # insert into word1   (from left, just updated)
                    prev        # replace             (diagonal)
                )
            prev = temp        # slide diagonal forward

    return dp[n]

# Slow motion on word1="horse", word2="ros":
# init dp=[0,1,2,3]  (converting '' to 'ros')
# i=1(h): dp=[1,1,2,3]  h≠r: 1+min(1,1,0)=1; h≠o: ...; h≠s: ...
# i=2(o): dp=[2,2,1,2]  o≠r: ...; o==o: take diag dp[i-1][j-1]=1
# i=3(r): dp=[3,2,2,2]
# i=4(s): dp=[4,3,3,2]  s==s: take diag=2
# i=5(e): dp=[5,4,4,3]
# return 3

def test_harness(fn):
    tests = [
        ("horse", "ros", 3),
        ("intention", "execution", 5),
        ("", "", 0),
        ("abc", "abc", 0),
        ("", "abc", 3),
        ("abc", "", 3),
        ("a", "b", 1),
    ]
    passed = 0
    for *inputs, expected in tests:
        w1, w2 = inputs
        got = fn(w1, w2)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | '{w1}'→'{w2}' | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(min_distance)
print("min_distance defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Longest Common Subsequence — LC 1143

---

```
PROBLEM:
  Given two strings, return the length of their longest common subsequence.
  A subsequence keeps characters in order but may skip some.

TRICK:
  dp[i][j] = length of LCS of text1[0..i-1] and text2[0..j-1].
  If text1[i-1] == text2[j-1]: dp[i][j] = dp[i-1][j-1] + 1  (extend LCS)
  Else:                         dp[i][j] = max(dp[i-1][j], dp[i][j-1])  (skip one)
  Base: dp[0][j]=0 and dp[i][0]=0 (empty string has LCS=0 with anything).

SLOW MOTION TRACE on text1="abcde", text2="ace":

       ''  a   c   e
  ''  [ 0] [0] [0] [0]
  a   [ 0] [1] [1] [1]   a==a: dp[0][0]+1=1
  b   [ 0] [1] [1] [1]   b≠a: max(0,1)=1; b≠c: max(1,1)=1; b≠e: max(1,1)=1
  c   [ 0] [1] [2] [2]   c≠a: max(1,0)=1; c==c: dp[1][1]+1=2
  d   [ 0] [1] [2] [2]   d≠all: max of neighbors
  e   [ 0] [1] [2] [3]   e==e: dp[3][2]+1=3

  answer = dp[5][3] = 3  (LCS = "ace")

KEY INSIGHT:
  Match → extend diagonally (+1). No match → take the best of skip-one-from-either-side.
  LCS is the foundation for: Edit Distance, Diff algorithms, DNA alignment.

TIME:  O(m*n)
SPACE: O(n)
```

In [ ]:
def longest_common_subsequence(text1, text2):
    """
    LC 1143 — Longest Common Subsequence
    Approach: 2D DP — match extends diagonal; mismatch takes max of two skips.
    Args:
        text1 (str): first string.
        text2 (str): second string.
    Returns:
        int: length of longest common subsequence.
    Time:  O(m*n) — fill every cell of the m×n table
    Space: O(n)   — 1D rolling array (two rows at a time)
    """
    m, n = len(text1), len(text2)
    prev = [0] * (n + 1)   # dp[i-1][*] — previous row

    for i in range(1, m + 1):
        curr = [0] * (n + 1)   # dp[i][*] — current row
        for j in range(1, n + 1):
            if text1[i-1] == text2[j-1]:
                curr[j] = prev[j-1] + 1    # match: extend the LCS by 1 (from diagonal)
            else:
                curr[j] = max(prev[j], curr[j-1])  # no match: best of skip from either side
        prev = curr            # slide down — current row becomes next iteration's previous

    return prev[n]

# Slow motion on text1='abcde', text2='ace':
# i=1(a): j=1: a==a → curr[1]=prev[0]+1=1; j=2: a≠c → max(0,1)=1; j=3: a≠e → 1
# i=2(b): all mismatches → [0,1,1,1]
# i=3(c): j=2: c==c → prev[1]+1=1+1=2  → [0,1,2,2]
# i=4(d): all mismatches → [0,1,2,2]
# i=5(e): j=3: e==e → prev[2]+1=2+1=3  → [0,1,2,3]
# return 3

def test_harness(fn):
    tests = [
        ("abcde", "ace", 3),
        ("abc", "abc", 3),
        ("abc", "def", 0),
        ("", "abc", 0),
        ("bsbininm", "jmjkbkjkv", 1),
        ("oxcpqrsvwf", "shmtulqrypy", 2),
    ]
    passed = 0
    for *inputs, expected in tests:
        t1, t2 = inputs
        got = fn(t1, t2)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | '{t1}' vs '{t2}' | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(longest_common_subsequence)
print("longest_common_subsequence defined.")

<a id='9'></a>
## 9. The 2D DP Decision Map

```
QUESTION TYPE                    RECURRENCE                           LC PROBLEMS
────────────────────────────────────────────────────────────────────────────────
Unique grid paths                dp[i][j]=dp[i-1][j]+dp[i][j-1]       62
Grid paths with obstacles        same + obstacle→dp[i][j]=0            63
Min edit distance                match→diag; else 1+min(del,ins,rep)   72
Longest common subsequence       match→diag+1; else max(up,left)       1143
Longest palindromic subsequence  LCS(s, reversed(s))                   516
Min path sum in grid             dp[i][j]=min(up,left)+grid[i][j]      64
Interleaving strings             dp[i][j]=dp[i-1][j] or dp[i][j-1]    97
Distinct subsequences            dp[i][j]=dp[i-1][j]+dp[i-1][j-1]     115

SPACE OPTIMIZATION RULE:
  If dp[i][j] depends only on row i-1 and current row i → use 1D rolling array.
  Save prev = old row; build curr = new row; then prev = curr.
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for 2D DP:**

| Signal | What to Do |
|--------|------------|
| "paths in a grid" | dp[i][j] = from above + from left |
| "minimum edit / transform one to other" | Edit Distance shape |
| "longest common subsequence" | match→diagonal+1; else max of neighbors |
| "two strings, optimal alignment" | 2D dp with string indices as dimensions |
| "minimum path sum" | dp[i][j] = min(up,left) + cost |

**2. The O(m*n) operations — memorize these:**

```python
# GRID PATHS
dp = [1] * n
for i in range(1, m):
    for j in range(1, n):
        dp[j] += dp[j-1]    # above stays; add left

# EDIT DISTANCE (1D rolling)
dp = list(range(n+1))
for i in range(1, m+1):
    prev = dp[0]; dp[0] = i
    for j in range(1, n+1):
        temp = dp[j]
        dp[j] = prev if s1[i-1]==s2[j-1] else 1+min(temp,dp[j-1],prev)
        prev = temp

# LCS (2-row rolling)
prev = [0]*(n+1)
for i in range(1, m+1):
    curr = [0]*(n+1)
    for j in range(1, n+1):
        if s1[i-1]==s2[j-1]: curr[j]=prev[j-1]+1
        else: curr[j]=max(prev[j],curr[j-1])
    prev = curr
```

**3. Common templates:**

```python
# TEMPLATE: 2D DP WITH PADDING (string problems)
m, n = len(s1), len(s2)
dp = [[0]*(n+1) for _ in range(m+1)]
# base cases in row 0 and col 0
for i in range(m+1): dp[i][0] = i   # example: edit distance
for j in range(n+1): dp[0][j] = j
for i in range(1, m+1):
    for j in range(1, n+1):
        if s1[i-1] == s2[j-1]: dp[i][j] = dp[i-1][j-1]
        else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
return dp[m][n]
```

**4. Gotchas to not forget:**

```
❌  Accessing dp[i-1][j-1] without padding → IndexError at row/col 0
❌  Forgetting base cases: first row and column must be initialized manually
❌  Confusing dp[i][j] (using first i of s1 AND first j of s2) with dp[i-1][j-1]
❌  Edit distance: 'delete' = dp[i-1][j], 'insert' = dp[i][j-1], 'replace' = dp[i-1][j-1]
✅  Grid DP: all-1 first row and first column (only one way to reach any edge cell)
✅  With obstacle: 0 anywhere an obstacle is; also zero if obstacle blocks the first row/col
✅  Space optimize: if row i only depends on row i-1, swap to 1D rolling array
✅  LCS is a building block: edit distance = m+n-2*LCS (for delete/insert only)
```

## Summary Map

```
                    🗺️ 2D DYNAMIC PROGRAMMING
                              │
           ┌──────────────────┼──────────────────┐
           │                  │                  │
       GRID PATHS         TWO STRINGS         VARIANTS
       dp[i][j]=          (m × n table)           │
       above+left         s1 on rows           MIN PATH
       LC 62, 63          s2 on cols           SUM GRID
                              │                LC 64
                    ┌─────────┴─────────┐
                    │                   │
                EDIT DIST           LCS
                match→diag          match→diag+1
                else 1+min          else max(up,left)
                (del,ins,rep)       LC 1143
                LC 72

CORE RULE:
  Define dp[i][j] → write recurrence → set base cases → fill top-left to bottom-right.
  If dp[i][j] depends only on dp[i-1][*] and dp[i][*] → 1D rolling array saves space.
```

---
*End of 2D Dynamic Programming Master Guide — Sean Edition*